In [ ]:
%load_ext autoreload
%autoreload 2

In [110]:
import os
import sys

sys.path.append("..")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [111]:
CKP = "/restricteddata/ukaea/checkpoints/ifft_filtered/20250122_164556"
device = "cuda"

In [ ]:
import yaml
from omegaconf import OmegaConf

from utils import load_model_and_config
from models import get_model

from dataset.cyclone import CycloneDataset

cfg = OmegaConf.create(yaml.safe_load(open(f"{CKP}/config.yaml", "r")))

# traindata = CycloneDataset(
#     active_keys=cfg.dataset.active_keys,
#     path=cfg.dataset.path,
#     split="train",
#     random_seed=cfg.seed,
#     test_ratio=0.0,
#     normalization=cfg.dataset.normalization,
#     spatial_ifft=cfg.dataset.spatial_ifft,
#     in_memory=False,
#     input_seq_length=cfg.model.input_seq_length,
#     target_seq_length=cfg.model.bundle_seq_length,
#     trajectories=cfg.dataset.training_trajectories,
# )

data = CycloneDataset(
    active_keys=cfg.dataset.active_keys,
    split="val",
    random_seed=cfg.seed,
    normalization=None,
    spatial_ifft=False,
    in_memory=False,
    bundle_seq_length=cfg.model.bundle_seq_length,
    trajectories=cfg.dataset.validation_trajectories,
)

print(f"Val: {len(data)}")

In [ ]:
model = get_model(cfg, dataset=data)

model, _, _ = load_model_and_config(f"{CKP}/best.pth", model, device)

model = model.to(device)

In [88]:
import torch
import numpy as np

def invert_ifft(x):
    x = np.moveaxis(x, 0, -1).copy()
    x = x.view(dtype=np.complex64)
    x = np.fft.fftn(x, axes=(4, 5))
    x = np.stack([x.real, x.imag]).squeeze().astype("float32")
    return x


def normalize(x):
    # TODO proper normalization
    if cfg.dataset.normalization == "zscore":
        shift = x.mean((2, 3, 4, 5, 6), keepdims=True)
        scale = x.std((2, 3, 4, 5, 6), keepdims=True)
    elif cfg.dataset.normalization == "minmax":
        shift = x.min((2, 3, 4, 5, 6), keepdims=True)
        x_max = x.max((2, 3, 4, 5, 6), keepdims=True)
        scale = x_max - shift
    else:
        shift, scale = 0.0, 1.0
    
    return (x - shift) / (scale), shift, scale

In [105]:
ONESTEP = False
OUT_DIR = f"/system/user/publicdata/gyrokinetics/dumps/ifft_{'onestep' if ONESTEP else 'autoreg'}"
IDX_0 = 50
IDX_END = 70

In [ ]:
losses = []
sample_0 = data[IDX_0]
xt = sample_0.x.to(device).unsqueeze(0)
itg = sample_0.itg.to(device).unsqueeze(0)
timesteps = data.get_timesteps(torch.tensor([0], dtype=torch.long))
files = []

with torch.no_grad():
    for idx in range(IDX_0, IDX_END + 1):
        if ONESTEP:
            xt = data[idx].x.to(device).unsqueeze(0)

        ts = timesteps[:, idx].to(device)
        xt, shift, scale = normalize(xt)
        xt = model(xt, timestep=ts, itg=itg)
        # denormalize
        xt = xt * scale + shift

        b_xt = xt.squeeze(0).cpu().numpy()
        if cfg.dataset.spatial_ifft:
            b_xt = invert_ifft(b_xt)
        b_xt = b_xt.astype("float64").reshape(-1, order="F")
        # dump to file
        if OUT_DIR:
            ftarget = os.path.join(OUT_DIR, f"K{str(int(idx)).zfill(2)}")
            with open(ftarget, "wb") as f:
                files.append(ftarget)
                print(f"Writing file {ftarget}")
                f.write(b_xt)

In [107]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colormaps


def force_aspect(ax, aspect=1):
    im = ax.get_images()
    extent = im[0].get_extent()
    ax.set_aspect(abs((extent[1] - extent[0]) / (extent[3] - extent[2])) / aspect)


def plot4x4_animate(kfiles, dataset, title="", frames=5, start_idx=0, average=True):
    plt.rcParams["animation.html"] = "jshtml"
    plt.ioff()

    labels = [r"v_{par}", r"v_{\mu}", r"s", r"k_x", r"k_y"]
    comb = torch.combinations(torch.arange(5), 2).tolist()

    fig, ax = plt.subplots(5, 5, figsize=(30, 12))
    for i in range(5):
        for j in range(5):
            ax[i, j].remove()

    fig.suptitle(title)
    c_map = colormaps["RdBu"]
    preds = []
    for kfname in kfiles[:frames]:
        with open(kfname, "rb") as fid:
            kt = np.fromfile(fid, dtype=np.float64)
        kt = np.reshape(kt, (2, 32, 8, 16, 255, 32), order="F").astype("float32")
        preds.append(kt)

    def animate(t):
        sample = dataset[start_idx + t]
        
        xp = preds[t]
        xgt = sample.y.numpy()
        if cfg.dataset.spatial_ifft:
            xgt = invert_ifft(xgt)
        ts = sample.timestep.numpy().item()
        fig.suptitle(f"ts={ts:.2f}", fontsize=30)

        for i, j in comb:
            other = tuple([o for o in range(5) if o != i and o != j])

            if average:
                xp_plot = xp[0].mean(other)
                xgt_plot = xgt[0].mean(other)
            else:
                xp_plot = torch.tensor(xp[0]).permute(i, j, *other).numpy()[:, :, 0, 0, 0]
                xgt_plot = torch.tensor(xgt[0]).permute(i, j, *other).numpy()[:, :, 0, 0, 0]

            ax_ij = ax[i, j]
            pos = ax_ij.get_position()

            # create two new axes within the same space as the original subplot
            plot_width = 0.475 * pos.width
            left_margin = 0.0 * pos.width
            x_left_1 = pos.x0 + left_margin
            x_left_2 = x_left_1 + plot_width
            y = pos.y0
            h = pos.height
            ax1 = fig.add_axes([x_left_1, y, plot_width, h])
            ax2 = fig.add_axes([x_left_2, y, plot_width, h])

            # compute shared vmin and vmax
            vmin = min(xgt_plot.min(), xp_plot.min())
            vmax = max(xgt_plot.max(), xp_plot.max())

            ax1.matshow(xp_plot, cmap=c_map) #, vmin=vmin, vmax=vmax)
            ax2.matshow(xgt_plot, cmap=c_map) #, vmin=vmin, vmax=vmax)

            if i == 0:
                # Set axis labels
                ax1.set_title(r"PRED", fontsize=24)
                ax2.set_title(r"GT", fontsize=24)

            if j == 1 or (i == 1 and j == 2) or (i == 2 and j == 3) or (i == 3 and j == 4):
                ax_ij.set_ylabel(rf"${labels[i]}$", fontsize=14)

            if i == 3 or j == 1 or (i == 1 and j == 2) or (i == 2 and j == 3):
                ax_ij.set_xlabel(rf"${labels[j]}$", fontsize=14)

            # Remove axis ticks and labels
            ax1.set_xticks([])
            ax1.set_yticks([])
            ax2.set_xticks([])
            ax2.set_yticks([])
            ax1.tick_params(labelleft=False, labelbottom=False)
            ax2.tick_params(labelleft=False, labelbottom=False)
            # Force aspect ratio
            force_aspect(ax1)
            force_aspect(ax2)

    return animation.FuncAnimation(fig, animate, frames=frames)

In [108]:
ani = plot4x4_animate(files, dataset=data, frames=IDX_END - IDX_0, start_idx=IDX_0)
writer = animation.PillowWriter(fps=1, bitrate=600)
ani.save("onestep.gif" if ONESTEP else "autoreg.gif", writer=writer)